# Merging SMAP and GEDI Data — Central Virginia Tree Canopy Analysis

This notebook merges two satellite-derived datasets for the Central Virginia study area and produces a full suite of year-over-year visualizations:

| Dataset | Product | Version | Temporal Range | Spatial Resolution |
|---|---|---|---|---|
| **SMAP** | SPL3SMP_E | 006 | 2015-2023 | 9 km (region-wide aggregate) |
| **GEDI** | GEDI02_A | 002 | 2019-2025 | Shot-level, aggregated to county |

**Study jurisdictions:** Albemarle, Augusta, Charlottesville, Fluvanna, Greene, Louisa, Nelson, Rockingham  
**Overlap window for merged analysis:** 2019-2023 (5 years)

### Notebook Structure

| Section | Description |
|---|---|
| S1 | Install dependencies |
| S2 | Imports and configuration |
| S3 | Load data from S3 |
| S4 | Merge datasets |
| S5 | Visual 1 - SMAP soil moisture time series (region-wide) |
| S6 | Visual 2 - GEDI canopy height by county and year |
| S7 | Visual 3 - County-level dual-axis facet plot (SMAP + GEDI) |
| S8 | Visual 4 - Eco-hydrological scatter correlation |
| S9 | Temporal lag analysis (1-year and 2-year) |
| S10 | Visual 5 - Lagged cross-correlation regression panels |
| S11 | Visual 6 - Pearson r heatmap by county and lag |
| S12 | Visual 7 - Regional dual-axis trend line |
| S13 | Export JSON outputs to all three S3 destinations |
| S14 | Summary |

## S1 - Install Dependencies

In [ ]:
import subprocess, sys

packages = ["s3fs", "plotly", "kaleido", "scipy", "matplotlib", "seaborn"]
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages)
print("Dependencies installed.")

## S2 - Imports and Configuration

**S3 source paths (confirmed):**
- SMAP: `s3://central-virginia-tree-canopy-project/smap-annual-means/smap_annual_summary.csv`
- GEDI: `s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary.csv`

**JSON output destinations (all three receive every file):**
1. `s3://central-virginia-tree-canopy-project/dashboard-data/`
2. `s3://central-va-tree-canopy-dashboard/dashboard-data/`
3. `s3://central-va-tree-canopy-dashboard/data/`

In [ ]:
import os
import io
import json
import warnings
warnings.filterwarnings('ignore')

import boto3
import s3fs
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy import stats

# ── S3 source paths ───────────────────────────────────────────────────────────
S3_BUCKET   = "central-virginia-tree-canopy-project"
SMAP_S3_KEY = "smap-annual-means/smap_annual_summary.csv"
GEDI_S3_KEY = "gedi-county-summary/virginia_gedi_county_summary.csv"

# ── Three S3 output destinations (bucket, prefix) ────────────────────────────
S3_DESTINATIONS = [
    ("central-virginia-tree-canopy-project", "dashboard-data/"),
    ("central-va-tree-canopy-dashboard",     "dashboard-data/"),
    ("central-va-tree-canopy-dashboard",     "data/"),
]

# ── Local output directory (SageMaker) ────────────────────────────────────────
OUTPUT_DIR = "/home/ec2-user/SageMaker/smap_gedi_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Colour palette (consistent across all visuals) ───────────────────────────
SMAP_COLOUR    = "#1f77b4"   # blue  - soil moisture
GEDI_COLOUR    = "#2ca02c"   # green - canopy height
COUNTY_PALETTE = px.colors.qualitative.Set2

JURISDICTIONS = [
    "Albemarle", "Augusta", "Charlottesville",
    "Fluvanna", "Greene", "Louisa", "Nelson", "Rockingham"
]

print("Imports and configuration complete.")
print(f"  SMAP source  : s3://{S3_BUCKET}/{SMAP_S3_KEY}")
print(f"  GEDI source  : s3://{S3_BUCKET}/{GEDI_S3_KEY}")
print(f"  Output dir   : {OUTPUT_DIR}")
print("  JSON destinations:")
for b, p in S3_DESTINATIONS:
    print(f"    s3://{b}/{p}")

## S3 - Load Data from S3

In [ ]:
fs = s3fs.S3FileSystem(anon=False)

# Load SMAP annual summary (region-wide, one row per year)
smap_s3_path = f"s3://{S3_BUCKET}/{SMAP_S3_KEY}"
print(f"Loading SMAP from : {smap_s3_path}")
with fs.open(smap_s3_path, "rb") as f:
    smap_df = pd.read_csv(f)
smap_df['year'] = smap_df['year'].astype(int)

# Load GEDI county summary (per county per year)
gedi_s3_path = f"s3://{S3_BUCKET}/{GEDI_S3_KEY}"
print(f"Loading GEDI from : {gedi_s3_path}")
with fs.open(gedi_s3_path, "rb") as f:
    gedi_df = pd.read_csv(f)
gedi_df['year'] = gedi_df['year'].astype(int)

print(f"\nSMAP shape : {smap_df.shape}  | years : {sorted(smap_df['year'].unique())}")
print(f"GEDI shape : {gedi_df.shape}  | years : {sorted(gedi_df['year'].unique())}")
print(f"GEDI jurisdictions : {sorted(gedi_df['jurisdiction'].unique())}")

print("\n--- SMAP preview ---")
display(smap_df)
print("\n--- GEDI preview (first 10 rows) ---")
display(gedi_df.head(10))

## S4 - Merge Datasets

**Design note:** SMAP is a region-wide aggregate (one value per year, no county column).  
The merge broadcasts each year's SMAP value across all counties for that year — correct because the 9 km SMAP pixels cover the entire study-area footprint.  
The inner join retains only the **2019-2023** overlap window where both datasets have coverage.

In [ ]:
# Broadcast SMAP across all counties by merging on year only
merged_df = pd.merge(
    gedi_df,
    smap_df[['year', 'sm_mean', 'sm_min', 'sm_max', 'sm_std']],
    on='year',
    how='inner'
).rename(columns={'sm_mean': 'sm_mean_m3m3'})

# Ensure chronological order within each county (required for lag analysis)
merged_df = merged_df.sort_values(['jurisdiction', 'year']).reset_index(drop=True)

print(f"Merged shape  : {merged_df.shape}")
print(f"Overlap years : {sorted(merged_df['year'].unique())}")
print(f"Jurisdictions : {sorted(merged_df['jurisdiction'].unique())}")
display(merged_df)

## S5 - Visual 1: SMAP Soil Moisture Time Series (Region-Wide, 2015-2023)

The full SMAP record including pre-GEDI years provides hydrological context. The shaded green band marks the 2019-2023 overlap window used in the merged analysis.

In [ ]:
fig = go.Figure()

# Shaded uncertainty band (+/- 1 std)
fig.add_trace(go.Scatter(
    x=list(smap_df['year']) + list(smap_df['year'][::-1]),
    y=list(smap_df['sm_mean'] + smap_df['sm_std']) +
      list((smap_df['sm_mean'] - smap_df['sm_std'])[::-1]),
    fill='toself',
    fillcolor='rgba(31, 119, 180, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    name='+/- 1 Std Dev',
    hoverinfo='skip'
))

# Annual mean line
fig.add_trace(go.Scatter(
    x=smap_df['year'],
    y=smap_df['sm_mean'],
    mode='lines+markers',
    name='Annual Mean SM',
    line=dict(color=SMAP_COLOUR, width=2.5),
    marker=dict(size=8, symbol='circle'),
    hovertemplate='%{x}: %{y:.4f} m3/m3<extra></extra>'
))

# Highlight GEDI overlap window
fig.add_vrect(
    x0=2019, x1=2023,
    fillcolor='rgba(44, 160, 44, 0.08)',
    layer='below', line_width=0,
    annotation_text='GEDI overlap window',
    annotation_position='top left',
    annotation_font_size=11,
    annotation_font_color='#2ca02c'
)

fig.update_layout(
    title=dict(text='SMAP Surface Soil Moisture - Central Virginia Study Area (2015-2023)', font_size=15),
    xaxis=dict(title='Year', dtick=1, tickangle=-30),
    yaxis=dict(title='Volumetric Soil Moisture (m3/m3)'),
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1),
    template='plotly_white',
    height=420
)

fig.show()
fig.write_html(os.path.join(OUTPUT_DIR, 'visual1_smap_timeseries.html'))
print("Visual 1 saved.")

## S6 - Visual 2: GEDI Canopy Height by County and Year (2019-2025)

Grouped bar chart showing mean rh98 canopy height per jurisdiction per year across the full GEDI record.

In [ ]:
fig = px.bar(
    gedi_df.sort_values(['year', 'jurisdiction']),
    x='jurisdiction',
    y='canopy_height_mean_m',
    color='jurisdiction',
    facet_col='year',
    facet_col_wrap=4,
    color_discrete_sequence=COUNTY_PALETTE,
    labels={
        'canopy_height_mean_m': 'Mean Canopy Height rh98 (m)',
        'jurisdiction': 'Jurisdiction',
        'year': 'Year'
    },
    title='GEDI Mean Canopy Height (rh98) by Jurisdiction and Year - Central Virginia (2019-2025)',
    height=600
)

fig.update_xaxes(tickangle=45, tickfont_size=9)
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()
fig.write_html(os.path.join(OUTPUT_DIR, 'visual2_gedi_canopy_bars.html'))
print("Visual 2 saved.")

## S7 - Visual 3: County-Level Dual-Axis Facet Plot (SMAP + GEDI, 2019-2023)

Each panel shows one jurisdiction's soil moisture (blue, left axis) and canopy height (green, right axis) over the 5-year overlap window.

In [ ]:
jurisdictions = sorted(merged_df['jurisdiction'].unique())
n_cols = 4
n_rows = int(np.ceil(len(jurisdictions) / n_cols))

fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=jurisdictions,
    specs=[[{"secondary_y": True}] * n_cols for _ in range(n_rows)],
    horizontal_spacing=0.08,
    vertical_spacing=0.14
)

for idx, jur in enumerate(jurisdictions):
    row = idx // n_cols + 1
    col = idx % n_cols + 1
    sub = merged_df[merged_df['jurisdiction'] == jur].sort_values('year')
    show_legend = (idx == 0)

    fig.add_trace(
        go.Scatter(
            x=sub['year'], y=sub['sm_mean_m3m3'],
            mode='lines+markers',
            name='Soil Moisture (m3/m3)',
            line=dict(color=SMAP_COLOUR, width=2),
            marker=dict(symbol='circle', size=7),
            showlegend=show_legend,
            hovertemplate=jur + ' %{x}<br>SM: %{y:.4f} m3/m3<extra></extra>'
        ),
        row=row, col=col, secondary_y=False
    )

    fig.add_trace(
        go.Scatter(
            x=sub['year'], y=sub['canopy_height_mean_m'],
            mode='lines+markers',
            name='Canopy Height rh98 (m)',
            line=dict(color=GEDI_COLOUR, width=2, dash='dash'),
            marker=dict(symbol='square', size=7),
            showlegend=show_legend,
            hovertemplate=jur + ' %{x}<br>Height: %{y:.2f} m<extra></extra>'
        ),
        row=row, col=col, secondary_y=True
    )

fig.update_layout(
    title=dict(
        text='Central Virginia: Soil Moisture vs Canopy Height by Jurisdiction (2019-2023)',
        font_size=14
    ),
    template='plotly_white',
    height=n_rows * 280,
    legend=dict(orientation='h', yanchor='bottom', y=1.03, xanchor='right', x=1)
)

for i in range(1, len(jurisdictions) + 1):
    axis_key = 'yaxis' + str(i * 2) if i > 1 else 'yaxis2'
    fig.update_layout(**{axis_key: dict(tickfont=dict(color=GEDI_COLOUR))})

fig.show()
fig.write_html(os.path.join(OUTPUT_DIR, 'visual3_dual_axis_facet.html'))
print("Visual 3 saved.")

## S8 - Visual 4: Eco-Hydrological Scatter Correlation

Each point is one jurisdiction-year observation. Colour encodes jurisdiction; marker size encodes year (larger = more recent) to reveal temporal drift in the relationship.

In [ ]:
year_min = merged_df['year'].min()
year_max = merged_df['year'].max()
merged_df['marker_size'] = 8 + 10 * (merged_df['year'] - year_min) / max(year_max - year_min, 1)

fig = px.scatter(
    merged_df,
    x='sm_mean_m3m3',
    y='canopy_height_mean_m',
    color='jurisdiction',
    size='marker_size',
    text='year',
    color_discrete_sequence=COUNTY_PALETTE,
    labels={
        'sm_mean_m3m3': 'Volumetric Surface Soil Moisture (m3/m3)',
        'canopy_height_mean_m': 'Mean Canopy Height rh98 (m)',
        'jurisdiction': 'Jurisdiction'
    },
    title='Relationship Between Surface Soil Moisture and Canopy Height (2019-2023)',
    height=520
)

valid = merged_df.dropna(subset=['sm_mean_m3m3', 'canopy_height_mean_m'])
slope, intercept, r_val, p_val, _ = stats.linregress(
    valid['sm_mean_m3m3'], valid['canopy_height_mean_m']
)
x_range = np.linspace(valid['sm_mean_m3m3'].min(), valid['sm_mean_m3m3'].max(), 100)
fig.add_trace(go.Scatter(
    x=x_range,
    y=slope * x_range + intercept,
    mode='lines',
    name='OLS trend (r=' + f'{r_val:.2f}' + ', p=' + f'{p_val:.3f}' + ')',
    line=dict(color='grey', dash='dot', width=1.5)
))

fig.update_traces(
    textposition='top center', textfont_size=9,
    selector=dict(mode='markers+text')
)
fig.update_layout(template='plotly_white', legend=dict(orientation='v', x=1.01, y=1))
fig.show()
fig.write_html(os.path.join(OUTPUT_DIR, 'visual4_scatter_correlation.html'))
print(f"Visual 4 saved.  Overall r = {r_val:.3f}, p = {p_val:.4f}")

## S9 - Temporal Lag Analysis

Trees exhibit a delayed physiological response to hydrological stress: a severe drought in one year may produce measurable canopy dieback 1-2 years later. The `.shift()` operation is applied within each jurisdiction group to prevent values bleeding across county boundaries.

In [ ]:
merged_df['sm_mean_lag1'] = merged_df.groupby('jurisdiction')['sm_mean_m3m3'].shift(1)
merged_df['sm_mean_lag2'] = merged_df.groupby('jurisdiction')['sm_mean_m3m3'].shift(2)

lag_labels = {
    'sm_mean_m3m3': 'Same-Year (No Lag)',
    'sm_mean_lag1': '1-Year Hydrological Lag',
    'sm_mean_lag2': '2-Year Hydrological Lag'
}

print("Lag columns created. Sample (Albemarle):")
display(
    merged_df[merged_df['jurisdiction'] == 'Albemarle']
    [['year', 'jurisdiction', 'sm_mean_m3m3', 'sm_mean_lag1', 'sm_mean_lag2', 'canopy_height_mean_m']]
)

## S10 - Visual 5: Lagged Cross-Correlation Regression Panels

Three side-by-side panels compare same-year, 1-year-lag, and 2-year-lag relationships. The panel with the tightest regression band indicates the dominant hydrological buffer window.

In [ ]:
melted_df = merged_df.melt(
    id_vars=['year', 'jurisdiction', 'canopy_height_mean_m'],
    value_vars=['sm_mean_m3m3', 'sm_mean_lag1', 'sm_mean_lag2'],
    var_name='lag_type',
    value_name='soil_moisture_value'
)
melted_df['lag_type'] = melted_df['lag_type'].map(lag_labels)
melted_df = melted_df.dropna(subset=['soil_moisture_value', 'canopy_height_mean_m'])

lag_order = ['Same-Year (No Lag)', '1-Year Hydrological Lag', '2-Year Hydrological Lag']
fig = make_subplots(rows=1, cols=3, subplot_titles=lag_order, horizontal_spacing=0.08)

for col_idx, lag_label in enumerate(lag_order, 1):
    sub = melted_df[melted_df['lag_type'] == lag_label]

    for j_idx, jur in enumerate(sorted(sub['jurisdiction'].unique())):
        jur_data = sub[sub['jurisdiction'] == jur]
        show_legend = (col_idx == 1)
        fig.add_trace(
            go.Scatter(
                x=jur_data['soil_moisture_value'],
                y=jur_data['canopy_height_mean_m'],
                mode='markers',
                name=jur,
                marker=dict(color=COUNTY_PALETTE[j_idx % len(COUNTY_PALETTE)], size=9, opacity=0.8),
                showlegend=show_legend,
                hovertemplate=jur + '<br>SM: %{x:.4f}<br>Height: %{y:.2f} m<extra></extra>'
            ),
            row=1, col=col_idx
        )

    if len(sub) > 3:
        sl, ic, rv, pv, _ = stats.linregress(sub['soil_moisture_value'], sub['canopy_height_mean_m'])
        xr = np.linspace(sub['soil_moisture_value'].min(), sub['soil_moisture_value'].max(), 80)
        fig.add_trace(
            go.Scatter(
                x=xr, y=sl * xr + ic,
                mode='lines',
                name='OLS r=' + f'{rv:.2f}',
                line=dict(color='black', dash='dot', width=1.5),
                showlegend=False,
                hovertemplate='r=' + f'{rv:.3f}' + ', p=' + f'{pv:.3f}' + '<extra></extra>'
            ),
            row=1, col=col_idx
        )

fig.update_xaxes(title_text='Soil Moisture (m3/m3)')
fig.update_yaxes(title_text='Canopy Height rh98 (m)', col=1)
fig.update_layout(
    title=dict(text='Virginia Study Footprint: Canopy Height Response to Delayed Soil Moisture Stress', font_size=14),
    template='plotly_white',
    height=460,
    legend=dict(orientation='v', x=1.01, y=1)
)
fig.show()
fig.write_html(os.path.join(OUTPUT_DIR, 'visual5_lag_regression_panels.html'))
print("Visual 5 saved.")

## S11 - Visual 6: Pearson r Heatmap by County and Lag

Quantifies the strength of the soil-moisture to canopy-height relationship for each jurisdiction across the three lag configurations. The column with the highest absolute r value per row indicates the dominant hydrological buffer window for that jurisdiction.

In [ ]:
lag_cols    = ['sm_mean_m3m3', 'sm_mean_lag1', 'sm_mean_lag2']
lag_display = ['Same-Year', '1-Year Lag', '2-Year Lag']

r_matrix = []
p_matrix = []

print("--- Pearson Correlation Analysis ---")
for jur in sorted(merged_df['jurisdiction'].unique()):
    row_r = []
    row_p = []
    jur_data = merged_df[merged_df['jurisdiction'] == jur]
    for lag_col, lag_label in zip(lag_cols, lag_display):
        valid = jur_data[['canopy_height_mean_m', lag_col]].dropna()
        if len(valid) > 2:
            r, p = stats.pearsonr(valid['canopy_height_mean_m'], valid[lag_col])
        else:
            r, p = np.nan, np.nan
        row_r.append(round(r, 3) if not np.isnan(r) else np.nan)
        row_p.append(round(p, 3) if not np.isnan(p) else np.nan)
        if not np.isnan(r):
            print(f"  {jur:<18} {lag_label:<22} r = {r:+.3f}  p = {p:.3f}")
        else:
            print(f"  {jur:<18} {lag_label:<22} insufficient data")
    r_matrix.append(row_r)
    p_matrix.append(row_p)

r_df = pd.DataFrame(
    r_matrix,
    index=sorted(merged_df['jurisdiction'].unique()),
    columns=lag_display
)

fig = go.Figure(data=go.Heatmap(
    z=r_df.values,
    x=lag_display,
    y=r_df.index.tolist(),
    colorscale='RdBu',
    zmid=0, zmin=-1, zmax=1,
    text=[[f'{v:.3f}' if not np.isnan(v) else 'n/a' for v in row] for row in r_df.values],
    texttemplate='%{text}',
    colorbar=dict(title='Pearson r')
))
fig.update_layout(
    title=dict(text='Pearson r: Canopy Height vs Soil Moisture by Jurisdiction and Lag', font_size=14),
    xaxis_title='Lag Configuration',
    yaxis_title='Jurisdiction',
    template='plotly_white',
    height=420
)
fig.show()
fig.write_html(os.path.join(OUTPUT_DIR, 'visual6_pearson_heatmap.html'))
print("\nVisual 6 saved.")

## S12 - Visual 7: Regional Dual-Axis Eco-Hydrological Trend Line

Aggregates all jurisdictions into a single regional average per year and overlays SMAP and GEDI on a dual y-axis.

In [ ]:
regional = merged_df.groupby('year').agg(
    canopy_mean=('canopy_height_mean_m', 'mean'),
    sm_mean=('sm_mean_m3m3', 'mean')
).reset_index()

fig = make_subplots(specs=[[{"secondary_y": True}]])

fig.add_trace(
    go.Scatter(
        x=smap_df['year'], y=smap_df['sm_mean'],
        mode='lines+markers',
        name='SMAP Soil Moisture (full record)',
        line=dict(color=SMAP_COLOUR, width=2),
        marker=dict(symbol='circle', size=7),
        hovertemplate='%{x}: %{y:.4f} m3/m3<extra></extra>'
    ),
    secondary_y=False
)

fig.add_trace(
    go.Scatter(
        x=regional['year'], y=regional['canopy_mean'],
        mode='lines+markers',
        name='GEDI Mean Canopy Height (regional avg)',
        line=dict(color=GEDI_COLOUR, width=2, dash='dash'),
        marker=dict(symbol='square', size=8),
        hovertemplate='%{x}: %{y:.2f} m<extra></extra>'
    ),
    secondary_y=True
)

fig.add_vrect(
    x0=2019, x1=2023,
    fillcolor='rgba(44, 160, 44, 0.07)',
    layer='below', line_width=0,
    annotation_text='Merged analysis window',
    annotation_position='top left',
    annotation_font_size=10,
    annotation_font_color='#2ca02c'
)

fig.update_yaxes(
    title_text='Volumetric Soil Moisture (m3/m3)', secondary_y=False,
    title_font=dict(color=SMAP_COLOUR), tickfont=dict(color=SMAP_COLOUR)
)
fig.update_yaxes(
    title_text='Mean Canopy Height rh98 (m)', secondary_y=True,
    title_font=dict(color=GEDI_COLOUR), tickfont=dict(color=GEDI_COLOUR)
)
fig.update_layout(
    title=dict(text='Virginia Study Area: Year-Over-Year Eco-Hydrological Trend (SMAP + GEDI)', font_size=14),
    xaxis=dict(title='Year', dtick=1, tickangle=-30),
    template='plotly_white',
    height=440,
    legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
)
fig.show()
fig.write_html(os.path.join(OUTPUT_DIR, 'visual7_regional_dual_axis.html'))
print("Visual 7 saved.")

## S13 - Export JSON Outputs to All Three S3 Destinations

Every JSON file is written to all three destinations in a single loop:

| # | Bucket | Prefix |
|---|---|---|
| 1 | central-virginia-tree-canopy-project | dashboard-data/ |
| 2 | central-va-tree-canopy-dashboard | dashboard-data/ |
| 3 | central-va-tree-canopy-dashboard | data/ |

In [ ]:
s3_client = boto3.client('s3')

def upload_json_all(data, filename, description):
    """Serialize data to JSON and upload to all three S3 destinations."""
    body = json.dumps(data, allow_nan=False).encode('utf-8')
    for bucket, prefix in S3_DESTINATIONS:
        s3_client.put_object(
            Bucket=bucket,
            Key=prefix + filename,
            Body=body,
            ContentType='application/json'
        )
        print(f"  -> s3://{bucket}/{prefix}{filename}")
    print(f"  {description}: uploaded to {len(S3_DESTINATIONS)} destinations.\n")

print("Exporting JSON files to all S3 destinations...\n")

# 1. SMAP time series (full record, 2015-2023)
upload_json_all(
    smap_df.to_dict(orient='records'),
    'smap_timeseries.json',
    'SMAP time series'
)

# 2. GEDI county summary (full record, 2019-2025)
upload_json_all(
    gedi_df.to_dict(orient='records'),
    'gedi_county_summary.json',
    'GEDI county summary'
)

# 3. Merged dataset (overlap window, 2019-2023)
merged_export = merged_df.drop(columns=['marker_size'], errors='ignore')
upload_json_all(
    merged_export.where(merged_export.notna(), other=None).to_dict(orient='records'),
    'merged_smap_gedi.json',
    'Merged SMAP + GEDI'
)

# 4. Pearson r matrix
upload_json_all(
    r_df.reset_index().rename(columns={'index': 'jurisdiction'}).to_dict(orient='records'),
    'pearson_r_matrix.json',
    'Pearson r matrix'
)

print("All JSON exports complete.")

## S14 - Summary

### S3 Inputs

| Dataset | S3 Path |
|---|---|
| SMAP annual summary | s3://central-virginia-tree-canopy-project/smap-annual-means/smap_annual_summary.csv |
| GEDI county summary | s3://central-virginia-tree-canopy-project/gedi-county-summary/virginia_gedi_county_summary.csv |

### S3 Outputs - Written to All Three Destinations

| File | Dest 1 | Dest 2 | Dest 3 |
|---|---|---|---|
| smap_timeseries.json | central-virginia-tree-canopy-project/dashboard-data/ | central-va-tree-canopy-dashboard/dashboard-data/ | central-va-tree-canopy-dashboard/data/ |
| gedi_county_summary.json | central-virginia-tree-canopy-project/dashboard-data/ | central-va-tree-canopy-dashboard/dashboard-data/ | central-va-tree-canopy-dashboard/data/ |
| merged_smap_gedi.json | central-virginia-tree-canopy-project/dashboard-data/ | central-va-tree-canopy-dashboard/dashboard-data/ | central-va-tree-canopy-dashboard/data/ |
| pearson_r_matrix.json | central-virginia-tree-canopy-project/dashboard-data/ | central-va-tree-canopy-dashboard/dashboard-data/ | central-va-tree-canopy-dashboard/data/ |

### Key Analytical Patterns to Watch For

- **Positive correlation (r > 0.4):** Higher soil moisture translates into healthier, taller, or denser forest structural profiles.
- **Peak lag:** Whichever lag configuration registers the highest absolute r-value indicates the forest system's structural buffer window to local climate fluctuations.
- **2019 anomaly:** SMAP 2019 shows the highest soil moisture in the record (0.380 m3/m3) - check whether this corresponds to elevated canopy heights in 2020 or 2021 (1-2 year lag).
- **2023 decline:** Both SMAP (0.271 m3/m3, near record low) and several GEDI jurisdictions show reduced canopy heights - consistent with drought-induced canopy stress.